# 22c — the **Gregor/lmfit estimator** at truth: bin + Voigt least-squares, full distribution vs real

**Question (Anuar).** Our 22b cloud (produced by our *unbinned Lorentzian MLE*) has a **narrower FWHM
spread** and a **much smaller σ_fit** than the real data. Is that an artefact of *our* estimator — i.e.
does the experiment's own estimator (bin the PLE scan at 2.2 MHz/step, least-squares fit a
**Voigt + constant** with lmfit) reproduce the real distribution?

**This notebook.** Same as 22b (N_MC = 1000 runs per experiment, all at the **true** parameters), but the
estimator is **Gregor's, verbatim**: `src/fitting_lmfit.py` (VoigtModel + ConstantModel), 2.2 MHz bins,
≥3 counts/bin gate, `lmfit` least-squares. We then compare the resulting `(FWHM, σ_fit)` cloud — scatter
+ KDE contour — against the real data as points.

**Prior (21d).** 21d already compared our vs lmfit vs real on **medians** (N_SIM = 40): lmfit did *not*
close the gap — its σ **overshot** ~3–5× at high T (e.g. 3nW T100: real 1.02, lmfit 4.24, ours 0.24) and
its low-T FWHM pinned at a bound. 22c re-runs it at N = 1000 and looks at the **full distribution**,
which is where the spread question actually lives.

Not a gradient/optimisation notebook — pure forward estimator comparison.

## Panel
- **FIG (interactive, Plotly)** — per power (1nW, then 3nW): the **lmfit-estimated** cloud at truth
  (points + KDE contour) vs the **real data** as points; experiment dropdown; no animation.
- **TABLE** — per experiment: fit success rate, lmfit FWHM (median ± std) vs real, lmfit σ_fit (median)
  vs real err (median), and a 2-sample KS p-value on the FWHM distributions.

Figures render **inline only**; the notebook is executed **in place**.

In [1]:
# ============================================================
# 22c — imports
# ============================================================
import math, time, os, sys
import numpy as np
import torch
import matplotlib.pyplot as plt

torch.set_default_dtype(torch.float32)

for _p in [os.getcwd(), os.path.join(os.getcwd(), '..'), os.path.join(os.getcwd(), '..', '..')]:
    if os.path.isdir(os.path.join(_p, 'src')):
        sys.path.insert(0, _p); REPO_ROOT = _p; break
os.chdir(REPO_ROOT)

from src.samplers import draw_fixed_noise, build_photons
from src import fitting_lmfit as FL

import plotly
import plotly.graph_objects as go
import plotly.io as pio
from scipy.ndimage import gaussian_filter
from scipy.stats import ks_2samp

pio.templates.default = 'plotly_white'
pio.renderers.default = os.environ.get('PLOTLY_RENDERER', 'vscode')

print('Imports OK  |  plotly', plotly.__version__, '| lmfit', FL.lmfit.__version__)

Imports OK  |  plotly 7.1.0 | lmfit 1.3.4


In [2]:
# ============================================================
# EXPERIMENTS — true values from Gregor's fits (identical to 17f/16-series) + real data file
# ============================================================
EXPERIMENTS = [
    dict(name='1nW Trans05',  power='1nW', mu_true=9.393, sigma_prop=2.576, lam=2.232, gamma_true=8.5, n_target=61, data_file='data/raw_data/fwhm_1nW_240221/fwhm_1nW_240221SIL_Puppy_hindleg_red1nW_Top20nW_Trans05.txt'),
    dict(name='1nW Trans10',  power='1nW', mu_true=12.372, sigma_prop=3.445, lam=2.122, gamma_true=8.5, n_target=358, data_file='data/raw_data/fwhm_1nW_240221/fwhm_1nW_240221SIL_Puppy_hindleg_red1nW_Top20nW_Trans10.txt'),
    dict(name='1nW Trans20',  power='1nW', mu_true=17.316, sigma_prop=4.141, lam=2.286, gamma_true=8.5, n_target=1138, data_file='data/raw_data/fwhm_1nW_240221/fwhm_1nW_240221SIL_Puppy_hindleg_red1nW_Top20nW_Trans20.txt'),
    dict(name='1nW Trans40',  power='1nW', mu_true=38.405, sigma_prop=7.198, lam=2.351, gamma_true=8.5, n_target=2428, data_file='data/raw_data/fwhm_1nW_240221/fwhm_1nW_240221SIL_Puppy_hindleg_red1nW_Top20nW_Trans40.txt'),
    dict(name='1nW Trans60',  power='1nW', mu_true=61.374, sigma_prop=9.851, lam=2.593, gamma_true=8.5, n_target=2424, data_file='data/raw_data/fwhm_1nW_240221/fwhm_1nW_240221SIL_Puppy_hindleg_red1nW_Top20nW_Trans60.txt'),
    dict(name='1nW Trans80',  power='1nW', mu_true=79.365, sigma_prop=12.627, lam=2.758, gamma_true=8.5, n_target=2487, data_file='data/raw_data/fwhm_1nW_240221/fwhm_1nW_240221SIL_Puppy_hindleg_red1nW_Top20nW_Trans80.txt'),
    dict(name='1nW Trans100', power='1nW', mu_true=70.817, sigma_prop=17.221, lam=2.636, gamma_true=8.5, n_target=2455, data_file='data/raw_data/fwhm_1nW_240221/fwhm_1nW_240221SIL_Puppy_hindleg_red1nW_Top20nW_Trans100.txt'),
    dict(name='3nW Trans05',  power='3nW', mu_true=13.204, sigma_prop=3.724, lam=2.186, gamma_true=14.1, n_target=252, data_file='data/raw_data/fwhm_3nW_210221/fwhm_3nW_210221SIL_Puppy_hindleg_red3nW_Top20nW_Trans05.txt'),
    dict(name='3nW Trans10',  power='3nW', mu_true=24.476, sigma_prop=5.639, lam=2.158, gamma_true=14.1, n_target=1572, data_file='data/raw_data/fwhm_3nW_210221/fwhm_3nW_210221SIL_Puppy_hindleg_red3nW_Top20nW_Trans10.txt'),
    dict(name='3nW Trans20',  power='3nW', mu_true=34.279, sigma_prop=8.319, lam=2.264, gamma_true=14.1, n_target=2171, data_file='data/raw_data/fwhm_3nW_210221/fwhm_3nW_210221SIL_Puppy_hindleg_red3nW_Top20nW_Trans20.txt'),
    dict(name='3nW Trans40',  power='3nW', mu_true=84.892, sigma_prop=24.013, lam=2.475, gamma_true=14.1, n_target=3742, data_file='data/raw_data/fwhm_3nW_210221/fwhm_3nW_210221SIL_Puppy_hindleg_red3nW_Top20nW_Trans40.txt'),
    dict(name='3nW Trans60',  power='3nW', mu_true=103.203, sigma_prop=23.95, lam=2.741, gamma_true=14.1, n_target=2541, data_file='data/raw_data/fwhm_3nW_210221/fwhm_3nW_210221SIL_Puppy_hindleg_red3nW_Top20nW_Trans60.txt'),
    dict(name='3nW Trans80',  power='3nW', mu_true=137.537, sigma_prop=32.107, lam=2.911, gamma_true=14.1, n_target=2508, data_file='data/raw_data/fwhm_3nW_210221/fwhm_3nW_210221SIL_Puppy_hindleg_red3nW_Top20nW_Trans80.txt'),
    dict(name='3nW Trans100', power='3nW', mu_true=175.707, sigma_prop=40.975, lam=3.087, gamma_true=14.1, n_target=2516, data_file='data/raw_data/fwhm_3nW_210221/fwhm_3nW_210221SIL_Puppy_hindleg_red3nW_Top20nW_Trans100.txt'),
]
print(f'{len(EXPERIMENTS)} experiments configured')

14 experiments configured


In [3]:
# ============================================================
# 22c CONFIG — Monte-Carlo at the TRUE parameters, GREGOR's estimator
# ============================================================
N_MC   = 1000
SEED   = 42
BIN_WIDTH = 2.2     # MHz (paper: 2.2 MHz/step scan resolution)
WINDOW    = 75.0    # MHz detection window
MIN_COUNTS = 3      # paper: reject a scan unless some bin has >=3 counts

SMOKE = os.environ.get('NB_SMOKE') == '1'
if SMOKE:
    N_MC = 100
    EXPERIMENTS = EXPERIMENTS[:2]
    print('*** SMOKE RUN ***')

N_WORKERS = 4
print(f'config: N_MC={N_MC}, bin={BIN_WIDTH} MHz, min_counts={MIN_COUNTS}, workers={N_WORKERS}')

config: N_MC=1000, bin=2.2 MHz, min_counts=3, workers=4


In [4]:
# ============================================================
# Gregor's estimator, verbatim (from 21d): bin the scan, least-squares fit Voigt+constant via lmfit
# ============================================================
def lmfit_extract(photons_np, bin_width=BIN_WIDTH, window=WINDOW):
    """Gregor's estimator: bin the scan, least-squares fit a Voigt+constant, return (FWHM, stderr)."""
    lo, hi = -window / 2.0, window / 2.0
    edges = np.arange(lo, hi + bin_width, bin_width)
    counts, edges = np.histogram(photons_np, bins=edges)
    if counts.size == 0 or counts.max() < MIN_COUNTS:
        return None, None
    x = 0.5 * (edges[:-1] + edges[1:])
    m = FL.voigt
    p = m.make_params()
    p['amplitude'].set(value=max(float(counts.sum()), 1.0), min=0.0)
    p['center'].set(value=float(x[int(np.argmax(counts))]), min=lo, max=hi)
    p['sigma'].set(value=3.0, min=0.05, max=50.0)
    p['gamma'].set(value=3.0, min=0.05, max=100.0)
    p['c'].set(value=float(np.median(counts)), min=0.0)
    try:
        out = m.fit(counts, p, x=x)
    except Exception:
        return None, None
    fw = out.params['fwhm'].value
    se = out.params['fwhm'].stderr
    fw = float(fw) if fw is not None and np.isfinite(fw) else None
    se = float(se) if se is not None and np.isfinite(se) else None
    return fw, se

def _lmf_one(photons_np):
    fw, se = lmfit_extract(photons_np)
    return (np.nan if fw is None else fw, np.nan if se is None else se)

def make_photons(gamma, mu, sig, lam, rng):
    u, b, n = draw_fixed_noise(mu, sig, lam, rng)
    return build_photons(torch.tensor(float(gamma)), torch.as_tensor(u, dtype=torch.float32),
                         torch.as_tensor(b, dtype=torch.float32)).numpy()

def load_target(exp):
    d = np.genfromtxt(exp['data_file'])
    fwhm_mhz = d[:, 0] * 1000.0; err_mhz = d[:, 1] * 1000.0
    ok = ~np.isnan(fwhm_mhz) & ~np.isnan(err_mhz) & (fwhm_mhz > 0)
    filt = ok & (err_mhz / fwhm_mhz < 10.0)
    return fwhm_mhz[filt], err_mhz[filt]

import multiprocessing as _mp
from concurrent.futures import ProcessPoolExecutor as _PPE
def _init_worker(): pass
def _run(args): return _lmf_one(args)
print('lmfit extractor ready')

lmfit extractor ready


In [5]:
# ============================================================
# RUN — 1000 lmfit-estimated scans at truth, per experiment
# ============================================================
t0 = time.time(); mc = []
for exp in EXPERIMENTS:
    gamma_true, mu, sig, lam = exp['gamma_true'], exp['mu_true'], exp['sigma_prop'], exp['lam']
    rng = np.random.default_rng(SEED)
    scans = [make_photons(gamma_true, mu, sig, lam, rng) for _ in range(N_MC)]
    with _PPE(max_workers=N_WORKERS, mp_context=_mp.get_context('fork'), initializer=_init_worker) as pool:
        res = list(pool.map(_run, scans, chunksize=16))
    F = np.array([r[0] for r in res]); S = np.array([r[1] for r in res])
    okm = np.isfinite(F) & np.isfinite(S)   # 2-D cloud needs both FWHM and its stderr
    tf, ts = load_target(exp)
    mc.append(dict(exp=exp['name'], power=exp['power'], T=int(exp['name'].split('Trans')[1]),
                   sim_f=F[okm], sim_s=S[okm], n_ok=int(okm.sum()), n_tot=N_MC,
                   target_f=tf, target_s=ts))
    print(f"{exp['name']:>12}: lmfit ok {okm.sum():4d}/{N_MC} | FWHM med {np.nanmedian(F):6.2f} "
          f"(real med {np.median(tf):6.2f}) | err med {np.nanmedian(S):5.2f} (real {np.median(ts):5.2f})")
print(f'\nTotal: {(time.time()-t0)/60:.1f} min   |   N_MC = {N_MC}')

 1nW Trans05: lmfit ok  257/1000 | FWHM med   6.00 (real med   3.74) | err med  3.09 (real  4.63)


 1nW Trans10: lmfit ok  424/1000 | FWHM med   6.00 (real med   6.38) | err med  3.05 (real  2.30)


 1nW Trans20: lmfit ok  740/1000 | FWHM med   6.00 (real med  10.78) | err med  3.20 (real  1.65)


 1nW Trans40: lmfit ok  995/1000 | FWHM med  13.87 (real med  15.57) | err med  3.41 (real  1.19)


 1nW Trans60: lmfit ok  997/1000 | FWHM med  16.05 (real med  16.17) | err med  3.08 (real  0.86)


...

 3nW Trans20: lmfit ok  922/1000 | FWHM med   7.86 (real med  26.04) | err med  4.96 (real  2.73)


 3nW Trans40: lmfit ok  992/1000 | FWHM med  20.81 (real med  24.97) | err med  4.93 (real  1.32)


 3nW Trans60: lmfit ok  997/1000 | FWHM med  23.09 (real med  29.86) | err med  4.89 (real  1.55)


 3nW Trans80: lmfit ok  999/1000 | FWHM med  23.72 (real med  28.88) | err med  4.42 (real  1.24)


3nW Trans100: lmfit ok 1000/1000 | FWHM med  24.97 (real med  28.29) | err med  4.27 (real  1.02)

Total: 2.2 min   |   N_MC = 1000


In [6]:
# ============================================================
# TABLE — lmfit cloud vs real, per experiment (fit rate, FWHM dist, err, KS)
# ============================================================
print(f"{'exp':<12} {'ok/N':>7} | {'FWHM med±std (lmfit)':>22} {'FWHM med±std (real)':>22} | {'err med lm':>10} {'err med real':>12} {'KS p':>7}")
print('-' * 108)
for r in mc:
    F, S = r['sim_f'], r['sim_s']; rf, rs = r['target_f'], r['target_s']
    ks = ks_2samp(F, rf).pvalue if len(F) > 1 and len(rf) > 1 else float('nan')
    print(f"{r['exp']:<12} {r['n_ok']:>4}/{r['n_tot']:<3} | "
          f"{np.median(F):>10.2f} ± {np.std(F):<7.2f} {np.median(rf):>12.2f} ± {np.std(rf):<7.2f} | "
          f"{np.median(S):>10.2f} {np.median(rs):>12.2f} {ks:>7.3f}")
print('\n(FWHM/err in MHz; KS p = 2-sample KS on the FWHM distributions, lmfit vs real.)')

exp             ok/N |   FWHM med±std (lmfit)    FWHM med±std (real) | err med lm err med real    KS p
------------------------------------------------------------------------------------------------------------
1nW Trans05   257/1000 |       6.00 ± 3.16            3.74 ± 169.78  |       3.09         4.63   0.000
1nW Trans10   424/1000 |       6.00 ± 4.45            6.38 ± 24.76   |       3.05         2.30   0.000
1nW Trans20   740/1000 |       6.00 ± 6.11           10.78 ± 10.46   |       3.20         1.65   0.000
1nW Trans40   995/1000 |      13.85 ± 7.22           15.57 ± 7.59    |       3.41         1.19   0.000
1nW Trans60   997/1000 |      16.04 ± 6.45           16.17 ± 5.71    |       3.08         0.86   0.000
1nW Trans80   995/1000 |      16.17 ± 5.61           16.90 ± 5.29    |       2.69         0.76   0.000
1nW Trans100  997/1000 |      16.13 ± 5.91           17.01 ± 7.93    |       2.84         0.85   0.000
3nW Trans05   315/1000 |       6.00 ± 5.14           16.74 ± 33.82 

In [7]:
# ============================================================
# PLOTLY — lmfit-estimated cloud at TRUTH (points + KDE) vs real data (points); per power; no animation
# ============================================================
def _grid_range(f, s, pct=(1, 99), pad=0.12):
    f = np.asarray(f)[np.isfinite(f)]; s = np.asarray(s)[np.isfinite(s)]
    lo_f, hi_f = np.percentile(f, pct); lo_s, hi_s = np.percentile(s, pct)
    df, ds = max(hi_f - lo_f, 1e-6), max(hi_s - lo_s, 1e-6)
    return (lo_f - pad*df, hi_f + pad*df), (lo_s - pad*ds, hi_s + pad*ds)

def _density(cf, cs, xr, yr, nx=40, ny=40, smooth=1.3):
    H, xe, ye = np.histogram2d(cf, cs, bins=[nx, ny], range=[xr, yr])
    Z = gaussian_filter(H.T, smooth); Z = Z / max(Z.max(), 1e-12)
    return 0.5*(xe[:-1]+xe[1:]), 0.5*(ye[:-1]+ye[1:]), np.round(Z*100).astype(int)

def build_lmfit_figure(mc, power):
    rs = sorted([r for r in mc if r['power'] == power], key=lambda r: r['T'])
    if not rs:
        print(f'no {power} experiments -> figure skipped'); return None
    fig = go.Figure(); ranges = {}
    for r in rs:
        cf, cs = r['sim_f'], r['sim_s']; tf, ts = r['target_f'], r['target_s']
        xr, yr = _grid_range(np.concatenate([cf, tf]), np.concatenate([cs, ts]))
        ranges[r['exp']] = (xr, yr)
        xc, yc, Z = _density(cf, cs, xr, yr)
        fig.add_trace(go.Contour(x=xc, y=yc, z=Z, colorscale='Blues', showscale=False,
                                 contours=dict(showlines=False), opacity=0.85, name='lmfit KDE'))
        fig.add_trace(go.Scatter(x=cf, y=cs, mode='markers',
                                 marker=dict(color='#457b9d', size=4, opacity=0.35), name=f'lmfit draws (N={N_MC})'))
        fig.add_trace(go.Scatter(x=tf, y=ts, mode='markers',
                                 marker=dict(color='#e63946', size=6, opacity=0.9, symbol='diamond'), name='real data'))
    n = len(rs); buttons = []
    for k, r in enumerate(rs):
        vis = [False]*(3*n); vis[3*k:3*k+3] = [True, True, True]
        xr, yr = ranges[r['exp']]
        buttons.append(dict(label=r['exp'], method='update',
            args=[{'visible': vis},
                  {'title': f"{r['exp']}  —  lmfit (Gregor) cloud at TRUTH (ok {r['n_ok']}/{r['n_tot']}) vs real data",
                   'xaxis': {'range': [xr[0], xr[1]], 'title': 'FWHM (MHz)'},
                   'yaxis': {'range': [yr[0], yr[1]], 'title': 'σ_fit (MHz, lmfit stderr)'}}]))
    x0, y0 = ranges[rs[0]['exp']]
    fig.update_layout(height=680, margin=dict(t=90, r=20),
        title=f"{rs[0]['exp']}  —  lmfit (Gregor) cloud at TRUTH vs real data",
        xaxis=dict(range=[x0[0], x0[1]], title='FWHM (MHz)'),
        yaxis=dict(range=[y0[0], y0[1]], title='σ_fit (MHz, lmfit stderr)'),
        updatemenus=[dict(type='dropdown', direction='down', x=1.0, y=1.12, xanchor='right',
                          showactive=True, buttons=buttons, active=0)])
    for i in range(len(fig.data)):
        fig.data[i].visible = (i < 3)
    return fig

print('build_lmfit_figure ready')

build_lmfit_figure ready


In [8]:
fig_1nW = build_lmfit_figure(mc, '1nW')
if fig_1nW is not None: fig_1nW.show()

In [9]:
fig_3nW = build_lmfit_figure(mc, '3nW')
if fig_3nW is not None: fig_3nW.show()

## Verdict — the estimator DOES explain the FWHM **spread** (lmfit reproduces it); it does NOT explain σ_fit (lmfit overshoots)

Estimator = **Gregor's, verbatim** (`src/fitting_lmfit.py`: bin at 2.2 MHz, ≥3 counts/bin, lmfit
`VoigtModel + ConstantModel` least-squares). Run at truth, N_MC = 1000, 14 experiments, 2.1 min.
(Figures use the series-22 palette: blue = lmfit draws, red = real data, Blues KDE.)

**(a) FWHM spread — CONFIRMED: it is the estimator.** lmfit's `std` reproduces the real FWHM `std` at
mid/high T, where our unbinned-Lorentzian-MLE (22b) was ~2× too narrow:

| cell | lmfit std | real std | ours (22b) |
|---|---|---|---|
| 3nW T40 | 10.44 | 10.29 | 5.54 |
| 3nW T60 | 9.75 | 10.13 | 4.82 |
| 3nW T80 | 8.96 | 9.14 | 4.09 |
| 1nW T60 | 6.45 | 5.71 | 3.70 |
| 1nW T80 | 5.61 | 5.29 | 3.15 |

Within ~3–10 % at 3nW T40–80. So the narrow spread was an **estimator-efficiency artefact, not missing
physics** — the unbinned MLE sits at the Cramér–Rao floor and is simply too precise per scan.

**(b) FWHM centre — lmfit is *worse*.** Its median is low by ~5–12 % at high T (3nW T100: 24.97 vs 28.29;
1nW T100: 16.13 vs 17.01), and at low T it collapses onto its **6.0 MHz bound** (only 257/1000 scans pass
the ≥3-counts gate at 1nW T05, 315/1000 at 3nW T05). Our estimator matched the high-T median much better
(1nW T100: 18.14 vs real 18.13). So lmfit buys the spread at the cost of the central value.

**(c) σ_fit — NOT explained; lmfit overshoots.** lmfit's `err` median is 2.7–5.0 vs real 0.76–1.55 at
mid/high T (**3–5× too big**), where ours was 2–4× too small. **Neither estimator's σ matches the paper's**
— the real value sits between them. (Consistent with 21d's median comparison, now confirmed on the full
distribution.) This is the one gap that survives both estimators.

**(d) KS p = 0.000 everywhere** — even where `std` matches, the distribution *shape* differs (lmfit's low
bias + low-T pinning), so the agreement is on the second moment, not the whole law.

**Read.** Anuar's hypothesis is **half right, and the half that is right matters**: the FWHM **spread** is an
estimator effect → to match the observable we should adopt the experiment's own estimator (the
"reproduce-the-estimator" logic in the journal). The **σ_fit** is *not* explained by lmfit either (ours too
small, lmfit too big) → the paper's fit-error definition needs to be pinned down with Gregor. And low T is
unusable for the lmfit route (count gate + bound pinning). Gradient/backprop considerations can come next,
on top of this estimator choice.